In [1]:
import pandas as pd
import numpy as np

DATA_PATH = '../data/ml-100k/'

ratings = pd.read_csv(DATA_PATH + 'u.data',sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
ratings_sorted = ratings.sort_values(by='timestamp')

split_index = int(0.8 * len(ratings_sorted))
train = ratings_sorted.iloc[:split_index]
test = ratings_sorted.iloc[split_index:]

print(f'Train: {len(train)} оценок')
print(f'Test:  {len(test)} оценок')
print(f'Train период: {train["timestamp"].min()} — {train["timestamp"].max()}')
print(f'Test период:  {test["timestamp"].min()} — {test["timestamp"].max()}')

Train: 80000 оценок
Test:  20000 оценок
Train период: 874724710 — 889237269
Test период:  889237269 — 893286638


In [2]:
ratings.head(5)

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [3]:
from sklearn.metrics import root_mean_squared_error

global_mean = train['rating'].mean()
test_pred_global = np.full(len(test), global_mean)

rmse_global = root_mean_squared_error(test['rating'], test_pred_global)

print(f'Global mean: {global_mean:.3f}')
print(f'RMSE (global mean baseline): {rmse_global:.4f}')

Global mean: 3.518
RMSE (global mean baseline): 1.1191


In [4]:
user_means = train.groupby('user_id')['rating'].mean()
test_pred_user = test['user_id'].map(user_means).fillna(global_mean)
rmse_user =  root_mean_squared_error(test['rating'], test_pred_user)

item_means = train.groupby('item_id')['rating'].mean()
test_pred_item = test['item_id'].map(item_means).fillna(global_mean)
rmse_item = root_mean_squared_error(test['rating'], test_pred_item)

print(f'RMSE (global mean): {rmse_global:.4f}')
print(f'RMSE (user means baseline): {rmse_user:.4f}')
print(f'RMSE (item means baseline): {rmse_item:.4f}')

RMSE (global mean): 1.1191
RMSE (user means baseline): 1.1258
RMSE (item means baseline): 1.0367


**global_mean** бьёт **user_mean** потому что датасет разбит по времени и при предсказании многие юзеры становятся новыми по этому в user_mean алгоритме мы заполняем их значениями global_mean, + добавляется шум из-за того что некоторых мы делим так что в трейне остаётся 1-2 его оценки и средние по такой выборке делает только хуже. Проверим сколько юзеров появляется только в тесте. 

In [5]:
users_in_train = set(train['user_id'].unique())
users_in_test = set(test['user_id'].unique())
new_users = users_in_test - users_in_train

print(f'Юзеров в test: {len(users_in_test)}')
print(f'Из них новых (нет в train): {len(new_users)}')

# сколько оценок в test принадлежат новым юзерам
n_new_ratings = test['user_id'].isin(new_users).sum()
print(f'Оценок в test от новых юзеров: {n_new_ratings} из {len(test)}')

Юзеров в test: 301
Из них новых (нет в train): 192
Оценок в test от новых юзеров: 17048 из 20000


Проанализировав split, я получил, что 85% тестовых оценок принадлежат юзерам, отсутствующим в train. Это заранее объясняет, почему user-based подходы здесь проседают, а item-based и content-based устойчивее.

In [6]:
user_item_matrix = train.pivot_table(index='user_id', columns='item_id', values='rating')

print(f'Размер матрицы: {user_item_matrix.shape}')
print(f'Заполнено ячеек: {user_item_matrix.notna().sum().sum()}')
print(user_item_matrix.iloc[:5, :5])

Размер матрицы: (751, 1616)
Заполнено ячеек: 80000
item_id    1    2    3    4   5
user_id                        
1        5.0  3.0  4.0  3.0 NaN
2        4.0  NaN  NaN  NaN NaN
3        NaN  NaN  NaN  NaN NaN
5        4.0  3.0  NaN  NaN NaN
6        4.0  NaN  NaN  NaN NaN


In [7]:
matrix_filled = user_item_matrix.fillna(0)
matrix_filled.head(5)

item_id,1,2,3,4,5,6,7,8,9,10,...,1660,1662,1664,1671,1672,1675,1676,1677,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,0.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,4.0,0.0,0.0,0.0,0.0,0.0,2.0,4.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Заполнял нулями как простой baseline, понимая, что это искажает геометрию: ноль трактуется как оценка ниже минимальной, и модель путает "не смотрел" с "не понравилось". Улучшением будет - **mean-centering**, где ноль означает среднюю оценку юзера, то есть честное "нейтрально". Это превращает косинус в корреляцию Пирсона

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(matrix_filled)

print(f'Матрица похожести: {user_similarity.shape}')
print(f'Похожесть юзера на самого себя (должна быть ~1): {user_similarity[0][0]:.3f}')
print(f'Пример: похожесть юзера 0 и юзера 1: {user_similarity[0][1]:.3f}')


Матрица похожести: (751, 751)
Похожесть юзера на самого себя (должна быть ~1): 1.000
Пример: похожесть юзера 0 и юзера 1: 0.145


In [9]:
user_ids = user_item_matrix.index
item_ids = user_item_matrix.columns

user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}

def predict_rating(user_id, item_id, k=30):
    if user_id not in user_id_to_idx or item_id not in item_ids:
        return global_mean

    u_idx = user_id_to_idx[user_id]
    item_ratings = user_item_matrix[item_id]
    rated_by = item_ratings.dropna()

    if len(rated_by) == 0:
        return global_mean

    sims = []
    ratings_list = []
    for other_user_id, rating in rated_by.items():
        if other_user_id == user_id:
            continue 

        other_idx = user_id_to_idx[other_user_id]
        sim = user_similarity[u_idx][other_idx]
        sims.append(sim)
        ratings_list.append(rating)

    sims = np.array(sims)
    ratings_list = np.array(ratings_list)

    if len(sims) > k:
        top_k_idx = np.argsort(sims)[-k:]
        sims = sims[top_k_idx]
        ratings_list = ratings_list[top_k_idx]

    if sims.sum() == 0:
        return global_mean
    prediction = np.sum(sims * ratings_list) / np.sum(sims)

    return prediction

print(predict_rating(user_ids[0], item_ids[0])) # протестировал на одном объекте тестовой выборки

3.8380437572452504


In [10]:
predictions = []
actuals = []

for row in test.itertuples():
    pred = predict_rating(row.user_id, row.item_id, k=30)
    predictions.append(pred)
    actuals.append(row.rating)

predictions = np.array(predictions)
actuals = np.array(actuals)

rmse_userknn = root_mean_squared_error(actuals, predictions)
print(f'RMSE (user-based KNN): {rmse_userknn:.4f}')

RMSE (user-based KNN): 1.1034


**User-based CF дал RMSE 1.1034, не побив item-mean baseline**. Причина не в алгоритме, а в данных: при temporal split 85% тестовых юзеров новые для модели (она не видела их в трейне), и для них user-based откатывается к global mean. Это подтвердило мой анализ сплита и показало, что item-based подходы здесь структурно выгоднее

# Вывод по user-based cf: c разбиением по времени
 **Train/test split - temporal (по времени):**
Данные отсортированы по timestamp, первые 80% (ранние оценки) -> train,
последние 20% (поздние) -> test. Так модель учится на прошлом и
проверяется на будущем - как в реальном проде, где будущих оценок мы не знаем.
Случайный split дал бы утечку: модель подсмотрела бы будущее поведение юзера.

**Что именно предсказываем:**
 Для каждой такой пары истинная оценка известна, но спрятана от
модели - модель предсказывает её по train-данным, затем сравниваем с реальной
через RMSE.

**Важное свойство данных (cold start):**
При temporal split ~85% тестовых оценок принадлежат юзерам, которых нет в train.
Для них user-based CF не может найти соседей (пустой вектор) и откатывается на
global mean. Это заранее объясняет слабость user-based подходов на данном сплите
и мотивирует item-based / content-based методы.

**Baseline'ы (планка, которую должны бить сложные модели):**
- Global mean: RMSE 1.1191
- User mean:   RMSE 1.1258 (хуже global — следствие cold start)
- Item mean:   RMSE 1.0367 (лучший baseline)

**ВАЖНО:** В данном эксперименте тестовые данные были полностью недоступны и низкий скор на user-based coloborative filtering возникает из-за того как раз что в тестовых 85% юзеров новые, соответсвенно мы не знаем их вектора в пространстве и берём global_mean. В следющем эксперименте я буду давать модели все оценки юзера в тесте кроме предсказываемой и за счёт этого предполагается улучшение точности модели по RMSE.

In [11]:
train.to_csv('../data/processed_train.csv', index=False)
test.to_csv('../data/processed_test.csv', index=False)
print('train/test сохранены')

train/test сохранены
